# AIQuorum: Getting Started Guide

Welcome to the AIQuorum step-by-step guide. This notebook will walk you through setting up the environment, configuring your agents, and running a consensus workflow.

## 1. Environment Setup

First, we need to ensure we have the necessary API keys. AIQuorum uses **OpenRouter** by default.

If you don't have a key, get one at [openrouter.ai](https://openrouter.ai/).

In [1]:
import os
from dotenv import load_dotenv

# Create a .env file if it doesn't exist
env_path = '.env'
if not os.path.exists(env_path):
    print("Creating .env file...")
    api_key = input("Enter your OpenRouter API Key: ")
    with open(env_path, 'w') as f:
        f.write(f"OPENROUTER_API_KEY={api_key}")
    print(".env file created!")
else:
    print(".env file already exists.")

# Load the environment variables
load_dotenv()

# Verify key is loaded
if os.environ.get("OPENROUTER_API_KEY"):
    print("✅ API Key loaded successfully.")
else:
    print("❌ API Key not found. Please check your .env file.")

.env file already exists.
✅ API Key loaded successfully.


## 2. Configuring Logging

AIQuorum provides detailed logs about the workflow progress, agent confidence, and potential errors (which are automatically retried). Let's configure logging to see them in this notebook.

In [2]:
import logging
import sys

# Set up logging to print to stdout so we can see it in Jupyter
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S',
    stream=sys.stdout,
    force=True # Force reconfiguration if basicConfig was already called
)

print("Logging configured.")

Logging configured.


## 3. Defining Agents

In AIQuorum, an `Agent` is a persona powered by an LLM.

In [3]:
import sys
import os
# Add src to path if running from examples folder without installing
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from aiquorum.agents.llm import Agent

# Define two agents with contrasting roles
architect = Agent(
    name="Architect",
    instructions="You are a senior system architect. You prioritize scalability, clean interfaces, and robust design patterns.",
    model="mistralai/devstral-2512:free",
    monitor=True
)

security = Agent(
    name="SecOps",
    instructions="You are a security operations expert. You look for vulnerabilities, injection risks, and data leakage.",
    model="kwaipilot/kat-coder-pro:free",
    monitor=True
)

# New: Consolidator Agent
# This agent is responsible for synthesizing the final answer from the debate.
consolidator = Agent(
    name="Moderator",
    instructions="You are a neutral moderator. Your job is to read the debate between the other agents and synthesize a final, authoritative answer that takes into account all valid points raised.",
    model="nvidia/nemotron-nano-12b-v2-vl:free",
    monitor=True
)

agents = [architect, security]
print("Agents initialized.")

Agents initialized.


## 4. Configuring the Workflow

The `Workflow` engine works as follows:
1. **Parallel Execution**: All agents (Architect, SecOps) generate responses simultaneously.
2. **Peer Review**: In subsequent steps, agents act as *peers*, critiquing and improving upon the aggregated knowledge of the previous step.
3. **Consolidation**: Once the threshold is met or max steps reached, the `consolidator` (Moderator) provides the final output.

In [5]:
from aiquorum.workflow.engine import Workflow

# Initialize the workflow
workflow = Workflow(
    agents=agents,
    max_steps=3,
    confidence_threshold=0.9,
    consolidator=consolidator, # Pass the consolidator here
    monitor=True # Enable monitoring logs
)

prompt = "Design a data storage system for a high-frequency trading platform."

print(f"Starting workflow for prompt: '{prompt}'")

Starting workflow for prompt: 'Design a data storage system for a high-frequency trading platform.'


## 5. Running the Workflow

Watch the output for "Invoking..." log messages. You should see them interleaved, indicating parallel execution.

In [6]:
try:
    result = workflow.run(prompt)
    print("Workflow completed!")
except Exception as e:
    print(f"An error occurred: {e}")

09:23:30 - aiquorum.workflow.engine - INFO - Starting workflow with max_steps=3, defined_agents=2
09:23:30 - aiquorum.workflow.engine - INFO - Monitor: Step 0 triggered
09:23:30 - httpx - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
09:23:32 - httpx - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
09:23:48 - aiquorum.workflow.engine - INFO - Monitor: response from agent Architect: Designing a data storage system for a high-frequency trading (HFT) platform requires a focus on ultr..., confidence score of agent Architect: 0.5
09:23:51 - aiquorum.workflow.engine - INFO - Monitor: response from agent SecOps: I'll design a comprehensive data storage system for a high-frequency trading (HFT) platform, focusin..., confidence score of agent SecOps: 0.5
09:23:51 - aiquorum.workflow.engine - INFO - Monitor: Step 1 triggered
09:23:51 - aiquorum.workflow.engine - INFO - Monitor: Agent Architect is reviewing pee

## 6. Analyzing Results

The `final_response` is now the result of the consolidation step.

In [7]:
print(f"Total Steps Taken: {result.total_steps}")
print(f"Stopping Reason: {result.reason}")
print(f"Final Confidence: {result.final_confidence:.2f}")
print(f"\n=== FINAL CONSOLIDATED ANSWER ===\n{result.final_response}")

print("\n--- Context History (The Debate) ---")
for response in result.history:
    print(f"\n[Step {response.step_number}] {response.agent_name} (Conf: {response.confidence})")
    print("-" * 20)
    print(response.content[:300] + "...")

Total Steps Taken: 2
Stopping Reason: Confidence threshold met
Final Confidence: 0.94

=== FINAL CONSOLIDATED ANSWER ===
To design a high-performance, low-latency data storage system for a high-frequency trading (HFT) platform, we synthesize insights from peer responses and address their gaps:

1. **Multi-Layered Storage Architecture**  
   - **In-Memory Tier (Critical Data)**: Use **Redis** or **Aerospike** for real-time order execution and trade data. Strengths: microsecond-level access. Weakness: Limited to <100 GB cost-effectively.  
   - **Time-Series Tier (Historical Data)**: Deploy **InfluxDB** or **Amazon Timestream** for market data and analytics. Strengths: Time-centric compression. Weakness: Latency (ms/100s of µs) unsuited for direct trading logic.  
   - **Hybrid Tier (Balance)**: **Kdb+/Nanomsq** combines time-series efficiency with in-memory speed but requires specialized engineering.  

2. **Partitioning & Colocation**  
   - **Asset-Based Partitioning**: Split data by 